 Lowercasing

In [2]:
import pandas as pd

df = pd.read_csv("/content/fake_and_real_news.csv")

print("Before:")
print(df["Text"].head())

# Convert to lowercase
df["Text"] = df["Text"].str.lower()

print("\nAfter:")
print(df["Text"].head())

Before:
0     Top Trump Surrogate BRUTALLY Stabs Him In The...
1    U.S. conservative leader optimistic of common ...
2    Trump proposes U.S. tax overhaul, stirs concer...
3     Court Forces Ohio To Allow Millions Of Illega...
4    Democrats say Trump agrees to work on immigrat...
Name: Text, dtype: object

After:
0     top trump surrogate brutally stabs him in the...
1    u.s. conservative leader optimistic of common ...
2    trump proposes u.s. tax overhaul, stirs concer...
3     court forces ohio to allow millions of illega...
4    democrats say trump agrees to work on immigrat...
Name: Text, dtype: object


2. Removing Punctuation

In [3]:
import string
# Remove punctuation
df["Text"] = df["Text"].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))

#cleaned = text.translate(str.maketrans('', '', string.punctuation))

print("\nAfter cleaning:")
print(df["Text"].head())


After cleaning:
0     top trump surrogate brutally stabs him in the...
1    us conservative leader optimistic of common gr...
2    trump proposes us tax overhaul stirs concerns ...
3     court forces ohio to allow millions of illega...
4    democrats say trump agrees to work on immigrat...
Name: Text, dtype: object


4. Removing Stopwords

In [4]:
import pandas as pd
import nltk
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords

# Stopword list
stop_words = set(stopwords.words('english'))

# Function to remove stopwords
def remove_stopwords(Text):
    tokens = Text.split()  # convert sentence to words
    filtered = [word for word in tokens if word.lower() not in stop_words]
    return " ".join(filtered)

# Apply to dataset column
df["Text"] = df["Text"].apply(remove_stopwords)

# Show result
print(df["Text"].head())

0    top trump surrogate brutally stabs back ‘he’s ...
1    us conservative leader optimistic common groun...
2    trump proposes us tax overhaul stirs concerns ...
3    court forces ohio allow millions illegally pur...
4    democrats say trump agrees work immigration bi...
Name: Text, dtype: object


TF-IDF (TfidfVectorizer)

Preprocessing---All operation at a time

In [5]:
import string
import nltk
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    tokens = text.split()
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['cleaned'] = df['Text'].apply(preprocess)

print("Original :", df['Text'].iloc[0])
print("Cleaned  :", df['cleaned'].iloc[0])

Original : top trump surrogate brutally stabs back ‘he’s pathetic’ video looking though republican presidential candidate donald trump losing support even within ranks know things getting bad even top surrogates start turning exactly happened fox news newt gingrich called trump pathetic gingrich knows trump needs keep focus hillary clinton even remotely wants chance defeating however trump hurt feelings many republicans support sexual assault women turned including house speaker paul ryan rwi made trump lash partygingrich said fox news look first let say trump admire tried help much big trump little trump little trump frankly pathetic mean mad getting phone call trump referring fact paul ryan call congratulate debate probably win despite trump ego tells himgingrich also added donald trump one opponent name hillary clinton name paul ryan anybody else trump seem realize person mad truly worst enemy ultimately lead defeat one blame himselfwatch via politicofeatured photo joe raedlegetty i

Vectorization — TF-IDF

In [6]:
df.head()

,Text,label,cleaned
0,top trump surrogate brutally stabs back ‘he’s ...,Fake,top trump surrogate brutally stabs back ‘he’s ...
1,us conservative leader optimistic common groun...,Real,us conservative leader optimistic common groun...
2,trump proposes us tax overhaul stirs concerns ...,Real,trump proposes us tax overhaul stirs concerns ...
3,court forces ohio allow millions illegally pur...,Fake,court forces ohio allow millions illegally pur...
4,democrats say trump agrees work immigration bi...,Real,democrats say trump agrees work immigration bi...


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X = df['cleaned']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print("Training matrix shape:", X_train_vec.shape)
print("Testing matrix shape :", X_test_vec.shape)

Training matrix shape: (7920, 80811)
Testing matrix shape : (1980, 80811)


Train Naive Bayes Model

In [8]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_vec, y_train)

print("Model trained successfully.")

Model trained successfully.


 Evaluate the Model

In [9]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test_vec)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=['Fake', 'Real']))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9636

Confusion Matrix:
[[920  53]
 [ 19 988]]

Classification Report:
              precision    recall  f1-score   support

        Fake       0.98      0.95      0.96       973
        Real       0.95      0.98      0.96      1007

    accuracy                           0.96      1980
   macro avg       0.96      0.96      0.96      1980
weighted avg       0.96      0.96      0.96      1980



Logistic Regression

In [35]:
df = pd.read_csv("/content/fake_and_real_news.csv")

In [36]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X = vectorizer.fit_transform(df["Text"])
y = df["label"]

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [38]:
from sklearn.linear_model import LogisticRegression

In [39]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [40]:
y_pred = model.predict(X_test)

In [41]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(f"Precision: {precision_score(y_test, y_pred, pos_label='Real'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, pos_label='Real'):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred, pos_label='Real'):.4f}")

Accuracy: 0.9929292929292929

Confusion Matrix:
[[ 963   10]
 [   4 1003]]

Precision: 0.9901
Recall:    0.9960
F1 Score:  0.9931


In [43]:
import joblib

# Save model
joblib.dump(model, "best_model.pkl")

# Save vectorizer also
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']